# **Python Programming Group Project**

## 🇫🇷 Paris Real Estate Investment Analysis: ROI under Rent Control (L'encadrement des loyers)

### 1. Context and Problem Statement

The Paris real estate market is characterized by high demand and strict regulatory oversight, notably the "Encadrement des loyers" (Rent Control) mechanism. This regulation caps the maximum rent a landlord can charge for a property based on its geographic location (neighborhood/district), size, number of rooms, and construction era.

This project aims to provide data-driven insights for real estate investors by fusing three complex public datasets—property sales, energy performance, and rent zone regulations—to model potential investment returns.

### 2. Project Goal

The primary objective is to develop a robust, calculated estimate of the Cash-on-Cash Return on Investment (ROI) for recent apartment purchases in Paris.

Future Goal: To create an interactive interface that allows investors to visualize high-ROI transactions based on specific criteria (e.g., minimum surface, maximum purchase price, desired energy class).

### 3. Data Sources Used

DVF (Demandes de Valeurs Foncières) from Direction Générale des Finances Publiques (DGFIP)

DPE (Diagnostic de Performance Énergétique) from ADEME (Agence de la transition écologique)

Paris Rent Zones (KML files) from DRIHL (Direction Régionale et Interdépartementale de l'Hébergement et du Logement)

### 4. Key Assumptions and Hypotheses

Due to the limitations and complexity of public datasets, the analysis relies on several strong filtering assumptions:

**A. Investment Scope**

- Property Type: We only consider apartments (DVF: $\text{code\_type\_local} = 2$) sold via a standard "Vente" transaction. Commercial, land, or dependency sales are filtered out.

- Investment Strategy: All transactions are modeled as buy-to-let (rental investment) opportunities.

- Rental Status: We assume the property is rented out unfurnished, as this simplifies the application of the rent control framework.

- Time Period: We focus on recent sales available in the DVF dataset.

**B. Data Integrity and Merging**

- Data Completeness: We assume that the properties successfully linked between DVF and DPE (a $58\%$ match rate) are representative of the larger market.

- Address Standardization: We assume that the highly robust address standardization process (stripping accents, cleaning street numbers, etc.) successfully resolved all feasible linking errors.

- DPE/Rent Linkage: Energy Class Impact: The DPE energy class is used to adjust the allowed rent:

$\text{A, B, C}$: Qualifies for the $\text{Loyer\_Ref\_Majore}$ (120% of the reference rent).

$\text{D, E}$: Defaults to the $\text{Loyer\_Ref}$ (Reference rent).

$\text{F, G}$: Must be capped at the $\text{Loyer\_Ref\_Minore}$ (90% of the reference rent).

- Unmatched DPE: Transactions without a DPE match default to the conservative $\text{Loyer\_Ref}$. (These are later filtered out for the final ROI calculation for reliability).

**C. Financial Modeling (ROI Calculation)**

- Purchase Price: The $\text{valeur\_fonciere}$ (transaction price) is assumed to be the total property value for calculation purposes.

- Funding: A standard financing structure is assumed:

- Down Payment: $20\%$ of the property value ($\text{Cash Invested}$).

- Loan Amount: $80\%$ of the property value.

- Expenses (Simplified Annual Model):

    - Property Taxes: Flat $0.5\%$ of the property value annually.

    - Mortgage Interest: Flat $4.0\%$ annual interest rate on the loan amount (ignoring amortization for a simple cash-on-cash estimate).

    - Operating Costs: $15\%$ of the annual gross rent is deducted to cover vacancies, management fees, and maintenance.


# Step 0: Environment Setup and Dependencies

## 🎯 Goal

The objective of this initial step is to:

Install all necessary Python libraries (packages) required for data manipulation, geospatial operations, geocoding, and web requests.

Import these libraries into the current Python environment.

Define global variables for the base URL of the DRIHL KML data and the local directory where the files will be stored.

Create the local storage directory (paris_kml_data/) if it doesn't already exist.

This ensures a stable, organized foundation before we proceed with data acquisition (Step 1).

## 📦 Dependencies

This project relies heavily on the following key libraries:

requests: For making HTTP requests to download the KML files.

pandas: The standard library for efficient data manipulation (used primarily for the final address dataset).

geopandas: Essential for reading KML files, handling geographic shapes (polygons), and performing spatial joins.

shapely: Used internally by geopandas to define and manipulate geometric objects (Points, Polygons).

geopy: For geocoding—converting street addresses into geographical coordinates (Latitude and Longitude).

tqdm: To display progress bars during file downloads, improving user experience.

In [ ]:
# =========================================================
# STEP -1: INSTALL ALL REQUIRED DEPENDENCIES
# (Run this ONCE when setting up your environment)
# =========================================================

%pip install --upgrade pip

%pip install \
    requests \
    pandas \
    numpy \
    geopandas \
    shapely \
    fiona \
    geopy \
    tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 44.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [ ]:
# =========================================================
# STEP 0: ENVIRONMENT SETUP & IMPORTS
# =========================================================

import os
import requests
import pandas as pd

# Geospatial libraries
import geopandas as gpd
from shapely.geometry import Point # Used for address geocoding
from fiona.drvsupport import supported_drivers

# Geocoding library
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# To track progress during long operations (like downloading)
from tqdm import tqdm

print("All required libraries imported successfully.")
print(f"geopandas version: {gpd.__version__}")
print(f"pandas version: {pd.__version__}")

# Define the base URL structure for the KML files
BASE_URL = "http://www.referenceloyer.drihl.ile-de-france.developpement-durable.gouv.fr/paris/kml/"

# Define the local directory where KML files will be stored
KML_DIR = "paris_kml_data"

# Create the directory if it doesn't exist
if not os.path.exists(KML_DIR):
    os.makedirs(KML_DIR)
    print(f"Created directory: {KML_DIR}")
else:
    print(f"Directory {KML_DIR} already exists.")

All required libraries imported successfully.
geopandas version: 1.1.1
pandas version: 2.2.2
Created directory: paris_kml_data


# Step 1: Automated KML Generation and Download
## 🎯 Goal

To create a systematic function that iterates through all specified combinations of variables (Period, Rooms, Construction Era) for non-furnished properties and automatically downloads the corresponding KML file from the DRIHL website.

## 📝 Code Explanation

Define Variables: We establish Python lists for all the parameters we need to iterate over.

Periods: Start date (YYYY-MM-DD) for each annual rent reference cycle, starting from your required cutoff of July 1st, 2019, up to the future period of July 1st, 2025. This gives 7 periods.

Rooms: 1, 2, 3, and 4+ rooms (represented by 1, 2, 3, 4).

Construction Eras: The four distinct periods used on the website.

Generate URLs and Download: A function iterates through these lists, constructs the full URL using the defined BASE_URL, and uses the requests library to fetch the file content.

Naming Convention: Each downloaded file is named logically (e.g., 2025-07-01_3_1946-1970_non-meuble.kml) to easily identify its parameters during the data preparation phase (Step 2).

Tracking: We use tqdm to display a progress bar, which is useful when downloading a large number of files.

In [ ]:
# =========================================================
# STEP 1: AUTOMATED KML GENERATION AND DOWNLOAD
# =========================================================

def generate_and_download_kmls():
    """
    Generates all required KML URLs and downloads the files to the KML_DIR.
    Constraints applied: Non-furnished only, data from 2019-07-01 onwards.
    """
    # --- 1. Define Variable Parameters ---

    # Validity periods (start date of the annual cycle)
    # Starting from 2019-07-01 as per your requirement (2019-2020 to 2025-2026)
    periods = [
        "2019-07-01", "2020-07-01", "2021-07-01", "2022-07-01",
        "2023-07-01", "2024-07-01", "2025-07-01"
    ] # 7 periods

    # Number of main rooms (as coded in the URL)
    rooms = ["1", "2", "3", "4"] # 4 types

    # Construction periods (as coded in the URL)
    construction_eras = ["inf1946", "1946-1970", "1971-1990", "sup1990"] # 4 eras

    # Type of rental
    rental_type = "non-meuble"

    total_files = len(periods) * len(rooms) * len(construction_eras)
    # 7 periods * 4 rooms * 4 eras = 112 files

    print(f"Total files to process: {total_files}")

    # --- 2. Generate and Download ---

    all_parameters = [
        (p, r, c)
        for p in periods
        for r in rooms
        for c in construction_eras
    ]

    success_count = 0

    # Use tqdm for a clear progress bar
    for period, room, construction in tqdm(all_parameters, desc="Downloading KMLs"):

        # Build the specific filename component of the URL
        # e.g., drihl_medianes_3_1946-1970_non-meuble.kml
        filename_url = f"drihl_medianes_{room}_{construction}_{rental_type}.kml"

        # Construct the full URL for the file
        full_url = f"{BASE_URL}{period}/{filename_url}"

        # Define the local path where the file will be saved
        # e.g., paris_kml_data/2025-07-01_3_1946-1970_non-meuble.kml
        local_filename = f"{period.replace('-', '')}_{room}_{construction}_{rental_type}.kml"
        local_path = os.path.join(KML_DIR, local_filename)

        # Check if the file already exists locally to avoid re-downloading
        if os.path.exists(local_path):
            # print(f"Skipping: {local_filename} already exists.")
            success_count += 1
            continue

        try:
            # Send an HTTP GET request to the URL
            response = requests.get(full_url, timeout=10)
            response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)

            # Write the content of the response to the local file
            with open(local_path, 'wb') as f:
                f.write(response.content)

            success_count += 1

        except requests.exceptions.RequestException as e:
            tqdm.write(f"ERROR downloading {local_filename} from {full_url}: {e}")

    print("\n--- Download Summary ---")
    print(f"Attempted: {total_files} files.")
    print(f"Successfully downloaded/found locally: {success_count} files.")

    if success_count == total_files:
        print("SUCCESS! All required KML files are now in the 'paris_kml_data' directory.")
    else:
        print("Warning: Not all files were retrieved. Check the error logs above.")

    return success_count == total_files

# Execute the download function
download_successful = generate_and_download_kmls()

Total files to process: 112



--- Download Summary ---
Attempted: 112 files.
Successfully downloaded/found locally: 112 files.
SUCCESS! All required KML files are now in the 'paris_kml_data' directory.


# Step 2: Loading and Preparation of Geospatial Data

## 🎯 Goal

To read all the KML files from the paris_kml_data directory, extract the necessary rent data and parameters (period, rooms, construction), and consolidate everything into one robust GeoDataFrame (a geographic table) called Paris_Rent_Zones_GDF.

## 📝 Code Explanation

Iterate Files: We use os.listdir to find every KML file in the designated directory.

Extract Parameters: The filename (which we designed in Step 1) is parsed to extract the period, rooms, and construction era. This metadata is crucial for matching the rent to the right query later.

Read KML with geopandas: gpd.read_file(path, driver='KML') reads the geographic data, automatically converting the KML structure into a GeoDataFrame, where the geometry column contains the polygons for the DRIHL zones.

Rename/Clean Columns: The columns in the KML are often generic (e.g., Name, Description). We clean these up and extract the actual rent values.

Concatenation: We append the data from each individual KML to a list, which is then used to create the final, massive Paris_Rent_Zones_GDF.

In [ ]:
# =========================================================
# STEP 2: LOADING AND PREPARATION OF GEOSPATIAL DATA
# =========================================================

# Ensure this is imported at the top of your notebook/script
from fiona.drvsupport import supported_drivers

def prepare_geospatial_data_revised(kml_dir):
    """
    Reads all KML files, extracts parameters, cleans the data, and concatenates
    them into a single master GeoDataFrame, using the explicit KML fields.
    """
    all_gdfs = []

    # Enable the KML driver for geopandas (using your successful fix)
    supported_drivers['KML'] = 'r'

    kml_files = [f for f in os.listdir(kml_dir) if f.endswith('.kml')]

    print(f"Starting preparation of {len(kml_files)} KML files...")

    for filename in tqdm(kml_files, desc="Processing KML files"):

        # --- 1. Extract Parameters from Filename (Remains the same) ---
        parts = filename.replace('.kml', '').split('_')
        if len(parts) != 4: continue

        period_str = parts[0]
        rooms_str = parts[1]
        construction_str = parts[2]
        period_start_date = f"{period_str[:4]}-{period_str[4:6]}-{period_str[6:]}"

        file_path = os.path.join(kml_dir, filename)

        try:
            # Read the KML file into a temporary GeoDataFrame (t_gdf)
            # The structure ensures 'ref', 'refmaj', 'refmin' are loaded as columns.
            t_gdf = gpd.read_file(file_path, driver='KML')

            # --- 2. Clean and Map Columns ---

            # We map the explicit KML fields to our desired column names
            t_gdf = t_gdf.rename(columns={
                'nameZone': 'DRIHL_Zone_ID',
                'ref': 'Loyer_Ref',
                'refmaj': 'Loyer_Ref_Majore',
                'refmin': 'Loyer_Ref_Minore'
            })

            # Add the parameters extracted from the filename
            t_gdf['Period_Start'] = period_start_date
            t_gdf['Rooms'] = t_gdf['piece'].str.split(' ').str[0].astype(int) # Extract '3' from '3 pièces'
            t_gdf['Construction_Era'] = construction_str

            # Convert rent columns to numeric type
            t_gdf['Loyer_Ref_Minore'] = pd.to_numeric(t_gdf['Loyer_Ref_Minore'], errors='coerce')
            t_gdf['Loyer_Ref'] = pd.to_numeric(t_gdf['Loyer_Ref'], errors='coerce')
            t_gdf['Loyer_Ref_Majore'] = pd.to_numeric(t_gdf['Loyer_Ref_Majore'], errors='coerce')

            # Keep only the essential columns and the geometry
            t_gdf = t_gdf[['DRIHL_Zone_ID', 'Loyer_Ref_Minore', 'Loyer_Ref',
                           'Loyer_Ref_Majore', 'Period_Start', 'Rooms',
                           'Construction_Era', 'geometry']]

            all_gdfs.append(t_gdf)

        except Exception as e:
            tqdm.write(f"ERROR processing {filename}: {e}")
            continue

    # --- 3. Concatenation and Final Setup (Remains the same) ---
    if not all_gdfs:
        print("No GeoDataFrames were successfully processed.")
        return None

    Paris_Rent_Zones_GDF = pd.concat(all_gdfs, ignore_index=True)

    # Ensure CRS is set
    Paris_Rent_Zones_GDF = gpd.GeoDataFrame(Paris_Rent_Zones_GDF,
                                            geometry='geometry',
                                            crs="EPSG:4326")

    print("\n--- Preparation Summary ---")
    print(f"Total rows in master GeoDataFrame: {len(Paris_Rent_Zones_GDF)}")
    print(f"Columns: {list(Paris_Rent_Zones_GDF.columns)}")
    print("Sample rows (first 5):")
    print(Paris_Rent_Zones_GDF.head())

    return Paris_Rent_Zones_GDF

# Execute the revised function
Paris_Rent_Zones_GDF = prepare_geospatial_data_revised(KML_DIR)

if Paris_Rent_Zones_GDF is not None:
    print("\nSUCCESS! The master GeoDataFrame is ready.")
    print("We have successfully combined all 112 KML files into one analytic table.")
    print("\nWe are ready to proceed to Step 3: Geocoding the Addresses.")

FINAL_GDF_VARIABLE_NAME = 'Paris_Rent_Zones_GDF'
OUTPUT_FILE_NAME = 'Paris_Rent_Zones_GDF.geojson'

# Assuming your final GeoDataFrame variable is named Paris_Rent_Zones_GDF
try:
    # Use 'to_file' to save the GeoDataFrame in GeoJSON format
    Paris_Rent_Zones_GDF.to_file(OUTPUT_FILE_NAME, driver="GeoJSON")
    print(f"\nSUCCESS! Master GeoDataFrame saved as '{OUTPUT_FILE_NAME}'.")
except NameError:
    print("\nERROR: Please check the exact name of your final GeoDataFrame variable from Step 2.")
    print("If it was named differently (e.g., 'gdf_final'), replace 'Paris_Rent_Zones_GDF' in the code above.")
except Exception as e:
    print(f"\nAn error occurred while saving the GeoDataFrame: {e}")

Starting preparation of 112 KML files...


Processing KML files: 100%|██████████| 112/112 [00:03<00:00, 30.66it/s]



--- Preparation Summary ---
Total rows in master GeoDataFrame: 8960
Columns: ['DRIHL_Zone_ID', 'Loyer_Ref_Minore', 'Loyer_Ref', 'Loyer_Ref_Majore', 'Period_Start', 'Rooms', 'Construction_Era', 'geometry']
Sample rows (first 5):
           DRIHL_Zone_ID  Loyer_Ref_Minore  Loyer_Ref  Loyer_Ref_Majore  \
0  Notre-Dame-des-Champs              16.9       24.1              28.9   
1   Saint-Thomas-d'Aquin              16.9       24.1              28.9   
2              Invalides              16.9       24.1              28.9   
3        Ecole-Militaire              16.9       24.1              28.9   
4           Gros-Caillou              16.9       24.1              28.9   

  Period_Start  Rooms Construction_Era  \
0   2019-07-01      4        1971-1990   
1   2019-07-01      4        1971-1990   
2   2019-07-01      4        1971-1990   
3   2019-07-01      4        1971-1990   
4   2019-07-01      4        1971-1990   

                                            geometry  
0  POLYGON Z

# Step 3: Data Ingestion & Cleaning

## 🎯 Strategy

### DVF Cleaning:

Filter: Keep only "Vente" (Sales) and "Appartement" (Apartments).

Paris Scope: Ensure we are only looking at zip codes starting with 75.

Essential Columns: Keep Price (valeur_fonciere), Date (date_mutation), Surface (surface_reelle_bati), Address (adresse_nom_voie, adresse_numero), and Coordinates (latitude, longitude).

Data Validity: Drop rows with zero surface or missing prices.

### DPE Cleaning (Prepared for the full file):

Target Columns: Extract annee_construction, classe_consommation_energie (the DPE letter), and address details.

The "DPE Logic": We will create a mapping column immediately:

A, B, C → Target_Rent = 'Majoré' (Premium)

D, E → Target_Rent = 'Ref' (Standard)

F, G → Target_Rent = 'Minoré' (Discounted)

### Address Standardization (Pre-work for Step 4):

To merge these two datasets later, we need a common key. We will create a join_key in both datasets formatted as: ZIPCODE_STREETNAME_NUMBER (e.g., 75001_RUE DE RIVOLI_10).

## 📝 Code Explanation

create_join_key: This is our "bridge". It strips punctuation and uppercase text to ensure that "Rue de la Paix" in one file matches "RUE DE LA PAIX" in the other.

DVF Cleaning: We were strict with the filters (nature_mutation == 'Vente' and code_type_local == 2) to ensure we are calculating ROI on residential apartments sold, not parking spots or commercial spaces.

DPE Logic: We implemented a specific logic (A-C = Majoré, D-E = Référence, F-G = Minoré) in the map_dpe_quality function. This creates a column Rent_Target_Column that tells us which DRIHL price to pick later.

In [ ]:
# =========================================================
# STEP 3: INGESTION AND CLEANING (DVF & DPE)
# =========================================================

import numpy as np
import pandas as pd
import re
import unicodedata

# Files
DVF_FILE = 'DVF_departement_75.csv'
DPE_FILE = 'DPE_avant_2021_departement_75.csv' # The full file you have locally

# --- Helper: Address Key Standardizer ---
def create_join_key(row, street_col, num_col, zip_col):
    """
    Creates a standardized key for joining: ZIP_STREET_NUM
    Removes accents (é -> e) and punctuation for maximum robustness.
    """
    try:
        # 1. Clean street name: Strip accents, uppercase, remove punctuation
        street = str(row[street_col]).strip()

        # Normalize (strip accents: 'rue de l'Église' -> 'rue de l Eglise')
        street = unicodedata.normalize('NFKD', street).encode('ascii', 'ignore').decode('utf-8')
        street = street.upper()
        # Remove all punctuation and symbols (e.g., RUE VIEILLE DU TEMPLE -> RUE VIEILLE DU TEMPLE)
        street = re.sub(r'[^\w\s]', '', street)

        # 2. Clean number: Use the cleaned number value (handled below in DVF/DPE cleaning)
        num = str(row[num_col]).split(' ')[0]

        # 3. Clean ZIP
        zip_code = str(row[zip_col]).replace('.0', '')

        return f"{zip_code}_{street}_{num}"
    except:
        return None

# --- 1. DVF Cleaning Function ---

def load_and_clean_dvf(filepath):
    print("Loading DVF data...")
    # Low_memory=False helps pandas read large files without guessing dtypes incorrectly
    df = pd.read_csv(filepath, low_memory=False)

    # === STEP 1: ROBUST TYPE CASTING ===
    # Coerce critical columns to numeric, filling non-numeric values with NaN/0 for safety
    df['code_type_local'] = pd.to_numeric(df['code_type_local'], errors='coerce').fillna(0).astype(int)
    df['valeur_fonciere'] = pd.to_numeric(df['valeur_fonciere'], errors='coerce')
    df['surface_reelle_bati'] = pd.to_numeric(df['surface_reelle_bati'], errors='coerce')

    # Clean and cast postal code to handle floats like '75020.0'
    df['code_postal'] = pd.to_numeric(df['code_postal'], errors='coerce').fillna(0).astype(int).astype(str)

    # === STEP 2: FILTERING ===

    # Filter 1: Paris Only (Code Postal starting with '75')
    df = df[df['code_postal'].str.startswith('75')].copy()

    # Filter 2: Sales (Vente) and Apartments (type 2)
    # Now that the column is integer, the comparison is safe
    df = df[df['nature_mutation'] == 'Vente'].copy()
    df = df[df['code_type_local'] == 2].copy()

    # Filter 3: Valid Surface and Price (using clean numeric columns)
    df = df[(df['valeur_fonciere'] > 1000) & (df['surface_reelle_bati'] > 9)].copy()

    # Select Essential Columns
    cols_to_keep = [
        'id_mutation', 'date_mutation', 'valeur_fonciere',
        'adresse_numero', 'adresse_nom_voie', 'code_postal',
        'nom_commune', 'nombre_pieces_principales', 'surface_reelle_bati',
        'latitude', 'longitude' # Crucial for Step 5
    ]
    # Filter columns that actually exist
    df = df[[col for col in cols_to_keep if col in df.columns]].copy()

    # --- STEP 3: Create Join Key (Unchanged logic, but now based on cleaned columns) ---
    print("Generating address keys for DVF...")
    df['join_key'] = df.apply(
        lambda x: create_join_key(x, 'adresse_nom_voie', 'adresse_numero', 'code_postal'),
        axis=1
    )

    # Convert Date
    df['date_mutation'] = pd.to_datetime(df['date_mutation'])

    print(f"DVF Cleaned: {len(df)} valid apartment sales found.")
    return df

# --- 2. DPE Cleaning Function ---
def load_and_clean_dpe(filepath):
    print("Loading DPE data (this might take a moment)...")

    # Reloading with error handling for separators
    try:
        df = pd.read_csv(filepath, low_memory=False, sep=',')
    except:
        # Fallback to semicolon if comma fails
        df = pd.read_csv(filepath, low_memory=False, sep=';')

    # ... (Filter and type casting code remains the same) ...
    target_cols = [
        'annee_construction', 'classe_consommation_energie',
        'nom_rue', 'numero_rue', 'code_postal'
    ]
    # Filter for columns that actually exist (intersection)
    existing_cols = [c for c in target_cols if c in df.columns]
    df = df[existing_cols].copy()

    # Filter: Valid Construction Date and DPE Rating
    df = df.dropna(subset=['annee_construction', 'classe_consommation_energie'])
    df = df[df['annee_construction'].astype(str).str.isnumeric()]
    df['annee_construction'] = df['annee_construction'].astype(int)
    df = df[(df['annee_construction'] > 1600) & (df['annee_construction'] <= 2025)]

    # --- DPE Logic Mapping (remains the same) ---
    def map_dpe_quality(letter):
        if str(letter) in ['A', 'B', 'C']:
            return 'Loyer_Ref_Majore'
        elif str(letter) in ['D', 'E']:
            return 'Loyer_Ref'
        elif str(letter) in ['F', 'G']:
            return 'Loyer_Ref_Minore'
        else:
            return 'Loyer_Ref'

    df['Rent_Target_Column'] = df['classe_consommation_energie'].apply(map_dpe_quality)

    # --- ADDRESS PARSING FIX (The Crux of the Problem) ---
    def fix_dpe_address_parsing(row):
        # We need two new columns for the join key: a clean street name and a clean number

        # 1. Normalize number column (handle NaN, float, etc.)
        num = pd.to_numeric(row['numero_rue'], errors='coerce')
        if not pd.isna(num):
            # If the number exists and is valid, use it
            row['numero_rue_clean'] = str(int(num)) # e.g., '32.0' -> '32'
            row['nom_rue_clean'] = str(row['nom_rue']).strip()
            return row

        # 2. If the number is NaN/empty (the common failure case), try to extract it from the street name
        street_name = str(row['nom_rue']).strip()

        # Regex: captures number (group 1) and the rest of the street name (group 2)
        match = re.match(r'^\s*(\d+)\s+([\w\s\'\-]+)', street_name, re.IGNORECASE)

        if match:
            # Success: extracted number and remaining street name
            row['numero_rue_clean'] = match.group(1)
            row['nom_rue_clean'] = match.group(2).strip()
        else:
            # Failure: number is missing or the format is too complex. Default to 'nan' for the number.
            row['numero_rue_clean'] = 'nan'
            row['nom_rue_clean'] = street_name

        return row

    # Apply the address fixing function
    print("Fixing DPE address parsing and preparing join columns...")
    df = df.apply(fix_dpe_address_parsing, axis=1)

    # --- Create Join Key using the new CLEAN columns ---
    print("Generating final address keys for DPE...")
    df['join_key'] = df.apply(
        lambda x: create_join_key(x, 'nom_rue_clean', 'numero_rue_clean', 'code_postal'),
        axis=1
    )

    # Drop temporary cleaning columns
    df = df.drop(columns=['nom_rue_clean', 'numero_rue_clean'], errors='ignore')

    # Deduplicate and save
    df = df.drop_duplicates(subset=['join_key'])

    print(f"DPE Cleaned: {len(df)} unique energy ratings found.")
    return df

# --- EXECUTION ---

import os

# 1. Process DVF
if os.path.exists(DVF_FILE):
    dvf_clean = load_and_clean_dvf(DVF_FILE)

    # Save the cleaned file to disk (RECOMMENDED)
    dvf_clean.to_csv('dvf_clean.csv', index=False)
    print("File 'dvf_clean.csv' created successfully.")

    print("\nDVF Head:")
    print(dvf_clean.head(3))
else:
    print(f"Error: DVF file {DVF_FILE} not found. Check the file name/path.")

# 2. Process DPE
if os.path.exists(DPE_FILE):
    dpe_clean = load_and_clean_dpe(DPE_FILE)

    # Save the cleaned file to disk (RECOMMENDED)
    dpe_clean.to_csv('dpe_clean.csv', index=False)
    print("File 'dpe_clean.csv' created successfully.")

    print("\nDPE Head:")
    print(dpe_clean.head(3))
else:
    print(f"Warning: DPE file {DPE_FILE} not found. Skipping DPE cleaning.")

# Step 4: The Grand Unification (Merging DVF and DPE)

## 🎯 Goal

Merge the cleaned DVF sales data with the cleaned DPE energy rating data using the standardized join_key. This will give every DVF transaction (which we assume is for a specific apartment in a building) the building's Construction Year and the Target Rent Category (Minoré, Référence, or Majoré) based on its DPE score.

## 📝 Code Explanation

Load Cleaned Data: We start by reloading the two CSV files we saved in Step 3.

Filter DPE: The DPE data is quite large, but we only need the key columns (annee_construction, Rent_Target_Column, join_key).

Perform Merge: We execute a left merge (how='left').

Left Side: The DVF sales dataset (all 168k transactions).

Right Side: The DPE dataset (the energy ratings).

Result: Every sale in the DVF keeps its data, and if a matching DPE key exists, the Construction Year and Rent Target columns are added. If no match is found, these columns will contain NaN (which we handle later).

Final Cleaning: We drop the temporary join_key column and handle missing values, which will form our final transactional dataset, Paris_Transactions_Merged.

In [ ]:
# =========================================================
# STEP 4: THE GRAND UNIFICATION (MERGING DVF AND DPE) - CORRECTED
# =========================================================

# Define file paths (assuming they were saved successfully in Step 3)
DVF_CLEAN_FILE = 'dvf_clean.csv'
DPE_CLEAN_FILE = 'dpe_clean.csv'
FINAL_TRANSACTIONS_FILE = 'paris_transactions_merged.csv'

def merge_dvf_dpe(dvf_path, dpe_path):
    print("Loading cleaned DVF and DPE files...")
    dvf_clean = pd.read_csv(dvf_path)
    dpe_clean = pd.read_csv(dpe_path)

    # --- 1. Prepare DPE for merging ---
    # We now include 'classe_consommation_energie' for the final output
    dpe_to_merge = dpe_clean[['join_key', 'annee_construction', 'Rent_Target_Column', 'classe_consommation_energie']].copy()

    # --- 2. Perform the Left Merge ---
    # We keep ALL DVF transactions (Left), and add DPE data where the keys match.
    print(f"Merging DVF ({len(dvf_clean)} rows) with DPE ({len(dpe_to_merge)} unique keys)...")

    Paris_Transactions_Merged = pd.merge(
        dvf_clean,
        dpe_to_merge,
        on='join_key',
        how='left'
    )

    print(f"Merge complete. Total transactions: {len(Paris_Transactions_Merged)}")

    # --- 3. Final Cleaning and Preparation ---

    # Fill missing Construction Year with 0 for safety (will be filtered out/grouped later)
    Paris_Transactions_Merged['annee_construction'] = Paris_Transactions_Merged['annee_construction'].fillna(0).astype(int)

    # Handle missing Rent_Target_Column (where no DPE match was found)
    Paris_Transactions_Merged['Rent_Target_Column'] = Paris_Transactions_Merged['Rent_Target_Column'].fillna('Loyer_Ref')

    # Handle missing DPE class (must be done AFTER the merge)
    Paris_Transactions_Merged['classe_consommation_energie'] = Paris_Transactions_Merged['classe_consommation_energie'].fillna('N') # N for Not available

    # Drop the temporary join key
    Paris_Transactions_Merged = Paris_Transactions_Merged.drop(columns=['join_key'])

    # Save the result
    Paris_Transactions_Merged.to_csv(FINAL_TRANSACTIONS_FILE, index=False)
    print(f"\nSUCCESS! Merged file saved as '{FINAL_TRANSACTIONS_FILE}'.")

    # Display summary
    match_count = Paris_Transactions_Merged['annee_construction'].astype(bool).sum()
    print(f"Transactions successfully linked with DPE/Construction data: {match_count} ({match_count/len(Paris_Transactions_Merged):.1%})")
    print("\nMerged Data Head:")
    print(Paris_Transactions_Merged.head())

    return Paris_Transactions_Merged

# Execute the merging function
Paris_Transactions_Merged = merge_dvf_dpe(DVF_CLEAN_FILE, DPE_CLEAN_FILE)

## Conclusion for Step 4

A 58% match rate for address-based joining between these two large public datasets is excellent. We have successfully enriched the majority of the DVF sales data with the crucial Construction Year and Rent Target columns.

We will proceed with the 97,958 linked transactions, as they represent the most complete and reliable subset of data for our ROI analysis. The 42% non-matched rows currently default to Loyer_Ref, which is a conservative approach, but we won't use them for the core ROI calculation in the next step.

# Step 5: Spatial Association and ROI Calculation

The final data file, paris_transactions_merged.csv, is now ready for the geospatial analysis and final calculation.

## 🎯 Goal

Spatial Join: Associate each transaction (via its latitude/longitude) with its geographic rent control zone (code_insee and rent columns) from the Paris_Rent_Zones_GDF GeoDataFrame created in Step 2.

Calculate ROI: Apply the formula to estimate the annual cash flow and ROI for each linked property.

## 📝 Code Explanation

This step requires the geospatial library geopandas, which must be installed locally.

In [ ]:
# =========================================================
# STEP 5: SPATIAL ASSOCIATION AND ROI CALCULATION (FINAL PREDICATE FIX)
# =========================================================

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import os

# File paths (unchanged)
FINAL_TRANSACTIONS_FILE = 'paris_transactions_merged.csv'
RENT_ZONES_GDF_FILE = 'Paris_Rent_Zones_GDF.geojson'
FINAL_ROI_FILE = 'Paris_ROI_Calculated.csv'

# Financial and market assumptions (unchanged)
ASSUMED_MORTGAGE_RATE = 0.04  # 4.0%
ASSUMED_DOWN_PAYMENT_PERCENT = 0.20 # 20% down payment
ASSUMED_PROPERTY_TAX_RATE = 0.005 # 0.5% of property value annually
RENTAL_EXPENSE_RATIO = 0.15 # 15% of rent for vacancies/management/maintenance

def calculate_roi(merged_transactions_file, rent_zones_file):
    print("Starting Step 5: Spatial Association and ROI Calculation...")

    # --- 1. Load Data ---
    df_transactions = pd.read_csv(merged_transactions_file)
    # Filter to only linked DPE rows AND ADD A SAMPLE FOR FASTER EXECUTION
    df_transactions = df_transactions[df_transactions['annee_construction'] > 0].copy()

    # TEMPORARY FIX: Sample a smaller portion of the data for faster execution
    # You can increase this number later if needed, but start small to ensure the process completes.
    if len(df_transactions) > 100000: # Adjust this threshold as needed
        df_transactions = df_transactions.sample(n=100000, random_state=42).copy()
    print(f"Loaded and sampled {len(df_transactions)} transactions (only linked DPE rows and sampled). Total after sampling: {len(df_transactions)}")

    gdf_rent_zones = gpd.read_file(rent_zones_file)

    # --- 2. Spatial Join ---
    geometry = [Point(xy) for xy in zip(df_transactions['longitude'], df_transactions['latitude'])]
    gdf_transactions = gpd.GeoDataFrame(df_transactions, geometry=geometry, crs=gdf_rent_zones.crs)

    print("Performing Spatial Join...")

    # **FINAL FIX:** Replaced 'op' with 'predicate'
    gdf_joined = gpd.sjoin(
        gdf_transactions,
        gdf_rent_zones[['DRIHL_Zone_ID', 'Loyer_Ref', 'Loyer_Ref_Minore', 'Loyer_Ref_Majore', 'geometry']],
        how='left',
        predicate='within' # <-- CORRECTED PARAMETER
    ).drop(columns=['index_right'])

    # --- 3. ROI Calculation ---

    # A. Select the correct rent based on DPE classification
    def select_max_rent(row):
        max_rent_per_sqm = row[row['Rent_Target_Column']]
        return max_rent_per_sqm * row['surface_reelle_bati']

    gdf_joined['Estimated_Monthly_Gross_Rent'] = gdf_joined.apply(select_max_rent, axis=1)

    # B. Calculate Annual Net Cash Flow

    gdf_joined['Annual_Gross_Rent'] = gdf_joined['Estimated_Monthly_Gross_Rent'] * 12
    property_value = gdf_joined['valeur_fonciere']

    # Expenses
    annual_taxes = property_value * ASSUMED_PROPERTY_TAX_RATE
    annual_expenses = gdf_joined['Annual_Gross_Rent'] * RENTAL_EXPENSE_RATIO
    loan_amount = property_value * (1 - ASSUMED_DOWN_PAYMENT_PERCENT)
    annual_interest = loan_amount * ASSUMED_MORTGAGE_RATE

    # Annual Net Cash Flow
    gdf_joined['Annual_Net_Cash_Flow'] = (
        gdf_joined['Annual_Gross_Rent']
        - annual_taxes
        - annual_expenses
        - annual_interest
    )

    # C. Calculate Cash-on-Cash Return (ROI)
    cash_invested = property_value * ASSUMED_DOWN_PAYMENT_PERCENT
    gdf_joined['Cash_on_Cash_ROI'] = (gdf_joined['Annual_Net_Cash_Flow'] / cash_invested) * 100

    # --- 4. Final Data Preparation ---

    final_cols = [
        'latitude', 'longitude', 'code_postal', 'surface_reelle_bati',
        'valeur_fonciere', 'annee_construction', 'classe_consommation_energie',
        'nombre_pieces_principales', # <--- ADDED THIS COLUMN
        'Estimated_Monthly_Gross_Rent', 'Annual_Net_Cash_Flow', 'Cash_on_Cash_ROI',
        'DRIHL_Zone_ID'
    ]
    df_final_roi = gdf_joined[final_cols].copy()

    print("Columns of df_final_roi before saving:")
    print(df_final_roi.columns.tolist())

    # Save the result
    df_final_roi.to_csv(FINAL_ROI_FILE, index=False)
    print(f"\nSUCCESS! Final ROI calculated and saved to '{FINAL_ROI_FILE}'.")

    return df_final_roi

# Execute the final calculation
if os.path.exists(RENT_ZONES_GDF_FILE) and os.path.exists(FINAL_TRANSACTIONS_FILE):
    Paris_ROI_Calculated = calculate_roi(FINAL_TRANSACTIONS_FILE, RENT_ZONES_GDF_FILE)
else:
    print(f"Error: Required files '{RENT_ZONES_GDF_FILE}' or '{FINAL_TRANSACTIONS_FILE}' not found. Ensure Step 2 and 4 ran successfully.")

## Step 6 – Investor Recommendation Tool & Interactive ROI Map

We now have a per-transaction dataset `Paris_ROI_Calculated.csv` containing:

- Location: `latitude`, `longitude`, `code_postal`, `DRIHL_Zone_ID`
- Property characteristics: `surface_reelle_bati`, `valeur_fonciere`,
  `annee_construction`, `classe_consommation_energie`,
  `nombre_pieces_principales`
- Financial indicators: `Estimated_Monthly_Gross_Rent`,
  `Annual_Net_Cash_Flow`, `Cash_on_Cash_ROI` (in **percent**)

The goal of this step is to turn these results into a simple decision-support
tool for an investor:

> The user specifies:
> - number of rooms,
> - maximum budget,
> - (optionally) arrondissement,
>
> and we display:
> - a **table** of the best opportunities (top ROI), and  
> - an **interactive map** of properties coloured by ROI  
>   (green = high, orange = medium, red = low).

This tool illustrates how our analysis could be used in practice to explore
investment opportunities in Paris under rent control.

In [ ]:
# STEP 6.2 – Load ROI dataset and prepare basic fields

import pandas as pd
import numpy as np

ROI_FILE = "Paris_ROI_Calculated.csv"

roi_df = pd.read_csv(ROI_FILE)
print(f"Loaded ROI file '{ROI_FILE}' with {len(roi_df)} rows.")

# Derive arrondissement from postal code (e.g. 75011 -> '11')
if "code_postal" in roi_df.columns:
    roi_df["arrondissement"] = roi_df["code_postal"].astype(str).str[-2:]
else:
    roi_df["arrondissement"] = np.nan

# Cash_on_Cash_ROI is stored as a percentage (e.g. 8 means 8%)
# Remove extreme outliers to keep the tool usable (1st–99th percentile)
if "Cash_on_Cash_ROI" in roi_df.columns:
    q_low, q_high = roi_df["Cash_on_Cash_ROI"].quantile([0.01, 0.99])
    roi_df = roi_df[
        (roi_df["Cash_on_Cash_ROI"] >= q_low) &
        (roi_df["Cash_on_Cash_ROI"] <= q_high)
    ].copy()
    print(f"After ROI outlier filtering: {len(roi_df)} rows.")

roi_df.head()

Loaded ROI file 'Paris_ROI_Calculated.csv' with 10957088 rows.
After ROI outlier filtering: 10737852 rows.


,latitude,longitude,code_postal,surface_reelle_bati,valeur_fonciere,annee_construction,classe_consommation_energie,Estimated_Monthly_Gross_Rent,Annual_Net_Cash_Flow,Cash_on_Cash_ROI,DRIHL_Zone_ID,arrondissement
1,48.884423,2.36959,75019,66.0,630000.0,1947,N,1471.8,-8297.64,-6.585429,Villette,19
2,48.884423,2.36959,75019,66.0,630000.0,1947,N,1386.0,-9172.80,-7.280000,Villette,19
3,48.884423,2.36959,75019,66.0,630000.0,1947,N,1696.2,-6008.76,-4.768857,Villette,19
4,48.884423,2.36959,75019,66.0,630000.0,1947,N,1518.0,-7826.40,-6.211429,Villette,19
5,48.884423,2.36959,75019,66.0,630000.0,1947,N,1247.4,-10586.52,-8.402000,Villette,19


In [ ]:
# STEP 6.3 – Filtering logic and ROI categorisation

# ROI thresholds in PERCENT (because Cash_on_Cash_ROI is already %)
GOOD_ROI_THRESHOLD = 8.0   # high ROI ≥ 8%
BAD_ROI_THRESHOLD  = 3.0   # low ROI ≤ 3%


def filter_properties(df, num_rooms, max_budget, arrondissement=None):
    """
    Filter the ROI dataset according to user preferences:
    - num_rooms: integer number of rooms
    - max_budget: maximum purchase price (valeur_fonciere)
    - arrondissement: '01'...'20' or None / 'All'
    """
    if df.empty:
        return df.copy()

    filtered = df.copy()

    # Filter by number of rooms
    if "nombre_pieces_principales" in filtered.columns:
        filtered = filtered[filtered["nombre_pieces_principales"] == num_rooms]

    # Filter by budget
    if "valeur_fonciere" in filtered.columns:
        filtered = filtered[filtered["valeur_fonciere"] <= max_budget]

    # Filter by arrondissement
    if arrondissement is not None and arrondissement != "All":
        filtered = filtered[filtered["arrondissement"] == arrondissement]

    # Drop rows without coordinates or ROI
    required_cols = ["latitude", "longitude", "Cash_on_Cash_ROI"]
    existing_required = [c for c in required_cols if c in filtered.columns]
    if existing_required:
        filtered = filtered.dropna(subset=existing_required)

    return filtered


def color_for_roi(roi_percent):
    """
    Assign a marker colour based on ROI (in percent).
    - Green: High ROI  (≥ GOOD_ROI_THRESHOLD)
    - Orange: Medium   (between BAD_ROI_THRESHOLD and GOOD_ROI_THRESHOLD)
    - Red: Low ROI     (≤ BAD_ROI_THRESHOLD)
    """
    if roi_percent >= GOOD_ROI_THRESHOLD:
        return "green"
    elif roi_percent <= BAD_ROI_THRESHOLD:
        return "red"
    else:
        return "orange"

In [ ]:
# Install dependencies for the interactive map & widgets
%pip install folium ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# STEP 6.4 – Function to display an interactive map of filtered properties

import folium
from folium.plugins import MarkerCluster

def show_investment_map(df_filtered, max_points=500):
    """
    Display an interactive Folium map of the filtered properties:
    - Each point is coloured according to its ROI.
    - A popup shows key info (price, surface, ROI, postal code).
    - For performance, we randomly sample up to max_points rows if needed.
    """
    if df_filtered.empty:
        print("No properties found with these criteria.")
        return None

    # Limit number of points for map performance
    if len(df_filtered) > max_points:
        df_vis = df_filtered.sample(max_points, random_state=42)
    else:
        df_vis = df_filtered.copy()

    # Center map on Paris
    paris_center = [48.8566, 2.3522]
    m = folium.Map(location=paris_center, zoom_start=12)

    # Cluster markers for readability
    marker_cluster = MarkerCluster().add_to(m)

    for _, row in df_vis.iterrows():
        roi_percent = row["Cash_on_Cash_ROI"]
        color = color_for_roi(roi_percent)

        popup_html = ""
        if "valeur_fonciere" in row:
            popup_html += f"<b>Price:</b> {row['valeur_fonciere']:.0f} €<br>"
        if "surface_reelle_bati" in row:
            popup_html += f"<b>Surface:</b> {row['surface_reelle_bati']} m²<br>"

        popup_html += f"<b>ROI:</b> {roi_percent:.1f}%<br>"

        if "code_postal" in row:
            popup_html += f"<b>Postal code:</b> {row['code_postal']}<br>"
        if "DRIHL_Zone_ID" in row:
            popup_html += f"<b>Zone:</b> {row['DRIHL_Zone_ID']}"

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=4,
            color=color,
            fill=True,
            fill_opacity=0.7,
            popup=popup_html
        ).add_to(marker_cluster)

    return m

In [ ]:
# STEP 6.5 – Interactive user interface (widgets)

import ipywidgets as widgets
from IPython.display import display, HTML

# Slider for number of rooms
room_widget = widgets.IntSlider(
    value=2,
    min=1,
    max=6,
    step=1,
    description="Rooms:",
    continuous_update=False
)

# Slider for budget
budget_widget = widgets.IntSlider(
    value=600_000,
    min=100_000,
    max=2_000_000,
    step=50_000,
    description="Budget (€):",
    continuous_update=False
)

# Dropdown for arrondissement
if roi_df["arrondissement"].notna().any():
    arr_options = ["All"] + sorted(roi_df["arrondissement"].dropna().unique().tolist())
else:
    arr_options = ["All"]

arr_widget = widgets.Dropdown(
    options=arr_options,
    value="All",
    description="Arrdt:"
)

# Button to trigger filtering and map display
button = widgets.Button(
    description="Show Investment Map",
    button_style="success",
    icon="map"
)

# Output area for table + map
output = widgets.Output()


def on_button_click(b):
    with output:
        output.clear_output()

        # Interpret "All" as no arrondissement filter
        arr_value = None if arr_widget.value == "All" else arr_widget.value

        # Apply filters
        df_filtered = filter_properties(
            roi_df,
            num_rooms=room_widget.value,
            max_budget=budget_widget.value,
            arrondissement=arr_value
        )

        if df_filtered.empty:
            print("No properties found for these filters.")
            return

        # Show top 10 properties by ROI
        cols_to_show = [
            "code_postal",
            "valeur_fonciere",
            "surface_reelle_bati",
            "Cash_on_Cash_ROI"
        ]
        existing_cols = [c for c in cols_to_show if c in df_filtered.columns]

        display(
            df_filtered[existing_cols]
            .sort_values("Cash_on_Cash_ROI", ascending=False)
            .head(10)
            .style.format(
                {
                    "valeur_fonciere": "{:,.0f}",
                    "Cash_on_Cash_ROI": "{:.1f}%"
                }
            )
        )

        # Show interactive map
        m = show_investment_map(df_filtered)
        if m is not None:
            # Try to display inside the notebook
            display(m)

            # Always save an HTML version for browser viewing
            html_path = "investment_map.html"
            m.save(html_path)
            display(
                HTML(
                    f"<p><b>Interactive map saved as "
                    f"<code>{html_path}</code>.</b><br>"
                    f"If the map does not appear above (trust issue in VS Code), "
                    f"open this file in your browser to view it.</p>"
                )
            )


button.on_click(on_button_click)

ui = widgets.VBox(
    [
        widgets.HBox([room_widget, budget_widget, arr_widget]),
        button,
        output
    ]
)

display(ui)

### Step 6 – Interpretation and Usage

The investor tool above lets a user explore the Paris housing market from an
investment perspective:

1. **Choose the number of rooms**  
   (e.g. 1–2 rooms for small rental units, 3–4 for family apartments).

2. **Set a maximum budget**  
   corresponding to the total purchase price (`valeur_fonciere`).

3. **Optionally select an arrondissement**  
   or keep “All” to search across Paris.

When the user clicks **“Show Investment Map”**:

- The dataset is filtered according to these criteria.
- The top 10 transactions by `Cash_on_Cash_ROI` are displayed in a table.
- An interactive map shows the corresponding properties:
  - 🟢 **Green** markers: high ROI (≥ 8%)  
  - 🟠 **Orange** markers: medium ROI (3–8%)  
  - 🔴 **Red** markers: low ROI (≤ 3%)

This demonstrates how our pipeline (DVF + DPE + DRIHL + financial model) can
be turned into a practical decision-support interface for investors, and it
also provides the target variable (`Cash_on_Cash_ROI`) that we will use in
the next step for predictive modelling.